# 04 - IA para la Automatización de la Respuesta a Incidentes

Este cuaderno implementa un sistema de triaje automático de incidentes y un motor de respuesta orquestada.

**Contenido:**
- Clasificación de severidad de incidentes con SVM
- Motor de respuesta automática (orquestación con Python)

## 5.1 Introducción

La IA puede integrarse en sistemas **SIEM** (Security Information and Event Management) para agilizar la identificación, contención y resolución de incidentes de seguridad, reduciendo el tiempo medio de respuesta (**MTTR**).

## 5.2 Triaje automático — Clasificación de severidad con SVM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------------------------------------------------------------
# Generar dataset sintético de incidentes si no existe
# Columnas: num_hosts_afectados, tipo_evento_cod,
#           bytes_exfiltrados, duracion_seg, privilegios_elevados
# Etiqueta: 0=bajo, 1=medio, 2=alto, 3=crítico
# ---------------------------------------------------------------
import os

if not os.path.exists('incident_data.csv'):
    print('Generando dataset sintético de incidentes...')
    rng = np.random.default_rng(42)
    n = 1200
    severity = rng.integers(0, 4, n)
    incidents_df = pd.DataFrame({
        'num_hosts_afectados' : severity * rng.integers(1, 5, n) + rng.integers(0, 3, n),
        'tipo_evento_cod'     : rng.integers(1, 10, n),
        'bytes_exfiltrados'   : severity * rng.integers(10000, 100000, n),
        'duracion_seg'        : rng.integers(10, 3600, n),
        'privilegios_elevados': rng.integers(0, 2, n),
        'severity'            : severity
    })
    incidents_df.to_csv('incident_data.csv', index=False)
    print(f'Dataset creado: {n} incidentes.')

# 1. Cargar datos de incidentes
incidents = pd.read_csv('incident_data.csv').dropna()
X = incidents.drop('severity', axis=1)
y = incidents['severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Pipeline: escalado + SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                probability=True, random_state=42))
])
pipeline.fit(X_train, y_train)

# 3. Evaluación
y_pred = pipeline.predict(X_test)
etiquetas = ['Bajo', 'Medio', 'Alto', 'Crítico']
print(classification_report(y_test, y_pred, target_names=etiquetas))

# 4. Probabilidades para un nuevo incidente
nuevo_incidente = pd.DataFrame([{
    'num_hosts_afectados' : 3,
    'tipo_evento_cod'     : 5,
    'bytes_exfiltrados'   : 102400,
    'duracion_seg'        : 120,
    'privilegios_elevados': 1
}])

proba = pipeline.predict_proba(nuevo_incidente)[0]
pred  = pipeline.predict(nuevo_incidente)[0]
print(f'\nNuevo incidente — Severidad predicha: {etiquetas[int(pred)]}')
for i, p in enumerate(proba):
    print(f'  P({etiquetas[i]}) = {p:.3f}')

## 5.3 Motor de respuesta automática (orquestación)

Una vez clasificado el incidente, Python puede orquestar respuestas automáticas integrándose con APIs de plataformas de seguridad (SOAR).

> **Nota:** Las llamadas HTTP están en modo simulado (`DRY_RUN=True`). Para conectar con una plataforma real, establece `DRY_RUN=False` y configura `BASE_URL` y el token de autorización.

In [ ]:
import requests
import logging
from dataclasses import dataclass
from enum import IntEnum

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(message)s'
)

# ---------------------------------------------------------------
# Modo simulado: no realiza llamadas HTTP reales
# ---------------------------------------------------------------
DRY_RUN  = True
BASE_URL = 'https://security-platform/api'  # reemplazar con URL real
AUTH_TOKEN = 'Bearer <TOKEN>'               # reemplazar con token real


class Severidad(IntEnum):
    BAJO    = 0
    MEDIO   = 1
    ALTO    = 2
    CRITICO = 3


@dataclass
class Incidente:
    id_sistema  : str
    ip_origen   : str
    severidad   : Severidad
    descripcion : str


def aislar_sistema(sistema_id: str) -> bool:
    """Envía orden de aislamiento de red al sistema comprometido."""
    if DRY_RUN:
        logging.info(f'[SIMULADO] Sistema {sistema_id} aislado correctamente.')
        return True
    try:
        resp = requests.post(
            f'{BASE_URL}/isolate/{sistema_id}',
            timeout=10,
            headers={'Authorization': AUTH_TOKEN}
        )
        resp.raise_for_status()
        logging.info(f'Sistema {sistema_id} aislado correctamente.')
        return True
    except requests.RequestException as e:
        logging.error(f'Error al aislar sistema {sistema_id}: {e}')
        return False


def bloquear_ip(ip: str) -> bool:
    """Agrega la IP al bloqueo en el firewall perimetral."""
    if DRY_RUN:
        logging.info(f'[SIMULADO] IP {ip} bloqueada en firewall.')
        return True
    try:
        resp = requests.post(
            f'{BASE_URL}/firewall/block',
            json={'ip': ip},
            timeout=10,
        )
        resp.raise_for_status()
        logging.info(f'IP {ip} bloqueada en firewall.')
        return True
    except requests.RequestException as e:
        logging.error(f'Error al bloquear IP {ip}: {e}')
        return False


def notificar_equipo(incidente: Incidente) -> None:
    """Envía alerta al equipo de respuesta (webhook)."""
    payload = {
        'text': (
            f'*ALERTA SEGURIDAD* Severidad: {incidente.severidad.name}\n'
            f'Sistema: {incidente.id_sistema}\n'
            f'IP: {incidente.ip_origen}\n'
            f'Descripción: {incidente.descripcion}'
        )
    }
    if DRY_RUN:
        logging.info(f'[SIMULADO] Notificación enviada al equipo:\n{payload["text"]}')
        return
    try:
        requests.post(f'{BASE_URL}/notify', json=payload, timeout=5)
        logging.info('Notificación enviada al equipo.')
    except requests.RequestException as e:
        logging.warning(f'Fallo en notificación: {e}')


def orquestar_respuesta(incidente: Incidente) -> None:
    """
    Selecciona y ejecuta acciones según la severidad del incidente.
    """
    logging.info(
        f'[Incidente] Sistema={incidente.id_sistema} '
        f'Severidad={incidente.severidad.name}'
    )

    if incidente.severidad == Severidad.BAJO:
        logging.info('Acción: registrar y monitorear.')

    elif incidente.severidad == Severidad.MEDIO:
        notificar_equipo(incidente)

    elif incidente.severidad == Severidad.ALTO:
        bloquear_ip(incidente.ip_origen)
        notificar_equipo(incidente)

    elif incidente.severidad == Severidad.CRITICO:
        aislar_sistema(incidente.id_sistema)
        bloquear_ip(incidente.ip_origen)
        notificar_equipo(incidente)


# Ejemplo de uso
inc = Incidente(
    id_sistema  = 'SRV-PROD-042',
    ip_origen   = '192.168.10.55',
    severidad   = Severidad.CRITICO,
    descripcion = 'Exfiltración masiva de datos detectada.'
)
orquestar_respuesta(inc)

## Resumen

- El pipeline SVM clasifica incidentes en 4 niveles de severidad.
- El motor de orquestación ejecuta acciones proporcionales: desde registrar hasta aislar sistemas y bloquear IPs.
- En producción, reemplaza `DRY_RUN=False` y configura las credenciales de tu plataforma SOAR.